#### 環境檢查

In [1]:
!nvidia-smi

Sun Aug 23 11:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### clone Github repo

In [3]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
!git clone https://{GITHUB_TOKEN}@github.com/Zhanzii9624/4GB-VRAM-RAG.git
%cd 4GB-VRAM-RAG

Cloning into '4GB-VRAM-RAG'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 84 (delta 31), reused 68 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 198.81 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/4GB-VRAM-RAG


### uv建立環境

In [4]:
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:" + os.environ['PATH']

!uv --version

uv 0.12.5 (x86_64-unknown-linux-gnu)


In [5]:
!uv sync

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 104 packages in 21.91s
Prepared 97 packages in 1m 40s
Installed 98 packages in 993ms
 + annotated-doc==0.0.5
 + anyio==4.14.2
 + aorus-rag==0.1.0 (from file:///content/4GB-VRAM-RAG)
 + asttokens==3.0.2
 + beautifulsoup4==4.15.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.4.2
 + comm==0.2.3
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.1
 + cuda-toolkit==13.0.3.0
 + debugpy==1.8.21
 + diskcache==5.6.3
 + executing==2.2.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + gdown==6.1.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.28.0
 + idna==3.19
 + ipykernel==7.3.0
 + ipython==9.16.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jieba==0.42.1
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.9.1
 + jupyter-core==5.9.1
 + llama-cpp-python==0.3.35
 + lxml==6.1.2
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplo

In [6]:
# uv sync一次裝好pyproject.toml
!uv sync

Resolved 104 packages in 1ms
Checked 98 packages in 1ms


In [7]:
# 確認llama-cpp-python(CUDA) & gdown
!uv run python -c "import llama_cpp; print('llama_cpp ok:', llama_cpp.__version__)"
!uv run python -c "import gdown; print('gdown ok:', gdown.__version__)"

llama_cpp ok: 0.3.35
gdown ok: 6.1.0


### 建立資料 parser, chunker, embedding

In [8]:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [9]:
!rm -f data/processed/*.json data/embeddings/*.npy
!uv run python rag/parser.py
!uv run python rag/chunker.py
!uv run python rag/embedding.py

Saved 21 records -> /content/4GB-VRAM-RAG/data/processed/specs.json
[chunker] Built 21 chunks.
[chunker] Saved 21 chunks → /content/4GB-VRAM-RAG/data/processed/chunks.json

--- Chunk 0 ---
[類別: 作業系統] [Variant: 全部型號 (BZH / BYH / BXH)]
作業系統 (Operating System): Windows 11 Pro / Windows 11 Home / UEFI Shell OS
本規格適用於全部型號 (BZH / BYH / BXH)。

--- Chunk 1 ---
[類別: 處理器] [Variant: 全部型號 (BZH / BYH / BXH)]
處理器 (CPU / Processor): Intel Core Ultra 9 Processor 275HX (36MB cache, up to 5.4GHz, 24 cores, 24 threads)
本規格適用於全部型號 (BZH / BYH / BXH)。

--- Chunk 2 ---
[類別: 顯示器] [Variant: 全部型號 (BZH / BYH / BXH)]
顯示器 (Display / Screen): 16-inch 16:10 OLED WQXGA (2560x1600), 240Hz, 1ms, DCI-P3 100%, 500nits peak, 1,000,000:1 contrast; NVIDIA G-SYNC; NVIDIA Advanced Optimus; VESA DisplayHDR True Black 500; VESA ClearMR 10000; Pantone Validated; TÜV Rheinland Low Blue Light; Dolby Vision
本規格適用於全部型號 (BZH / BYH / BXH)。
[embedding] Loading model: intfloat/multilingual-e5-small on cpu














/content/4GB-VRAM

### 下載模型

In [10]:
import gdown
from pathlib import Path

ROOT = Path.cwd()
LOCAL_MODEL = ROOT / "models" / "Qwen2.5-3B-Instruct-Q4_K_M.gguf"
LOCAL_MODEL.parent.mkdir(exist_ok=True)
print(f"LOCAL_MODEL: {LOCAL_MODEL}")
GDRIVE_FILE_ID = "1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW"  # downloaded from huggingface, stored in drive

if LOCAL_MODEL.exists():
    print(f"already here: {LOCAL_MODEL.stat().st_size / 1e9:.2f} GB")
else:
    print("downloading from Google Drive...")
    gdown.download(id=GDRIVE_FILE_ID, output=str(LOCAL_MODEL), quiet=False)
    print(f"done: {LOCAL_MODEL.stat().st_size / 1e9:.2f} GB")

LOCAL_MODEL: /content/4GB-VRAM-RAG/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
downloading from Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW
From (redirected): https://drive.google.com/uc?id=1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW&confirm=t&uuid=3d5d88b3-4d60-4df4-9ab8-3dfdd0be9f87
To: /content/4GB-VRAM-RAG/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
100%|██████████| 2.10G/2.10G [00:10<00:00, 207MB/s]

done: 2.10 GB


### Load LLM + Streaming測試 + TTFT/TPS測量

In [11]:
%%writefile scripts/demo_streaming.py
"""載入模型並跑一次 streaming 測試，量測 TTFT / TPS / VRAM。"""
import time
import subprocess
from inference.llama_engine import LlamaEngine
from rag.prompt import build_prompt

def get_vram_used_mb():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used",
        "--format=csv,noheader,nounits"
    ])
    return int(out.decode().strip())

print(f"VRAM before loading model: {get_vram_used_mb()} MB")

engine = LlamaEngine(n_gpu_layers=-1, n_ctx=2048, verbose=False)

print(f"VRAM after loading model:  {get_vram_used_mb()} MB")

dummy_ctx = [
    "[類別: 顯示晶片] [Variant: BZH]\n顯示晶片 (GPU / Graphics): NVIDIA GeForce RTX 5090 Laptop GPU, 24GB GDDR7, 175W Maximum Graphics Power with Dynamic Boost\n產品型號 BZH 專屬規格。"
]
prompt = build_prompt("BZH 的顯示晶片規格是什麼？", dummy_ctx)

start = time.perf_counter()
first_token_time = None
n_tokens = 0

for tok in engine.stream(prompt, max_new_tokens=150):
    if first_token_time is None:
        first_token_time = time.perf_counter()
    print(tok, end="", flush=True)
    n_tokens += 1

end = time.perf_counter()
print()
print(f"VRAM after generation:     {get_vram_used_mb()} MB")
print(f"TTFT: {(first_token_time - start) * 1000:.1f} ms")
print(f"TPS: {n_tokens / (end - first_token_time):.2f} tokens/sec")

Writing scripts/demo_streaming.py


In [12]:
!uv run python scripts/demo_streaming.py

VRAM before loading model: 0 MB
[llama_engine] Loading model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
[llama_engine] n_gpu_layers=-1, n_ctx=2048
[llama_engine] Model ready.
VRAM after loading model:  2315 MB
BZH 的顯示晶片規格如下：
- NVIDIA GeForce RTX 5090 Laptop GPU
- 24GB GDDR7
- 最大_graphics 功率為 175W，支援動態增強功能
VRAM after generation:     2347 MB
TTFT: 611.8 ms
TPS: 60.84 tokens/sec


In [13]:
# 量測VRAM 模型載入後、開始推論前
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader,nounits

Tesla T4, 0, 15360


### 重點檔案
vector+keyword fusion: hybrid_retriever.py

Prompt 拒答原則: rag/prompt.py

測試與題目: eval_benchmark.py

### 測試TTFT/TPS + 10-15題目

In [14]:
# 只速測ablation 不用gpu
!uv run python scripts/eval_benchmark.py --ablation

/content/4GB-VRAM-RAG/.venv/lib/python3.13/site-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/content/4GB-VRAM-RAG/.venv/lib/python3.13/site-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/content/4GB-VRAM-RAG/.venv/lib/python3.13/site-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[embedding] Loading model: intfloat/multilingual-e5-small on cpu

/content/4GB-VRAM-RAG/rag/embedding.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[embedding] Model loaded. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
[embedding] Model loaded. Embedding dim: 384
[embedding] Loaded embeddings shape: (21, 384)

=== Retrieval Ablation ===
TOP_K

In [15]:
!cat eval_results_ablation.json

{
  "top_k": 3,
  "excluded_ids": [
    "Q07",
    "Q08",
    "Q09",
    "Q10"
  ],
  "excluded_reason": "cross_variant 會被 pinning 機制覆蓋 alpha；abstain 無正確 chunk 可比對排名",
  "configs": {
    "vector-only": {
      "alpha": 1.0,
      "hit_rate": "8/11",
      "hit_rate_pct": 72.7,
      "avg_rank": 1,
      "per_question": [
        {
          "id": "Q01",
          "rank": null
        },
        {
          "id": "Q02",
          "rank": 1
        },
        {
          "id": "Q03",
          "rank": 1
        },
        {
          "id": "Q04",
          "rank": 1
        },
        {
          "id": "Q05",
          "rank": null
        },
        {
          "id": "Q06",
          "rank": 1
        },
        {
          "id": "Q11",
          "rank": 1
        },
        {
          "id": "Q12",
          "rank": 1
        },
        {
          "id": "Q13",
          "rank": null
        },
        {
          "id": "Q14",
          "rank": 1
        },
        {
          "id": "Q

In [16]:
!uv run python scripts/eval_benchmark.py

[embedding] Loading model: intfloat/multilingual-e5-small on cpu

/content/4GB-VRAM-RAG/rag/embedding.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[embedding] Model loaded. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
[embedding] Model loaded. Embedding dim: 384
[embedding] Loaded embeddings shape: (21, 384)
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.711 seconds.
Prefix dict has been built successfully.
[llama_engine] Loading model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
[llama_engine] n_gpu_layers=-1, n_ctx=2048
[llama_engine] Model ready.

=== [Q01] single_spec: 這台筆電的處理器（CPU）型號是什麼？ ===
embedding: 35.3 ms | retrieval: 31.3 ms | prefill(TTFT): 301.8 ms | TPS: 64.46
回答: 這台筆電的處理器（CPU）型號是 Intel Core Ultra 9 Processor 275HX。
檢索到 6 筆 chunk

=== [Q02] single_spec: BZH 型號的顯示晶片（GPU）規格是什麼？ ===
embedding: 33.8 ms | retrieval: 33.5 m